# 🧠 Emotion Detection — Complete Training & Model Saving Notebook
### Emotions NLP Dataset | Best model: Bidirectional LSTM (89.7%)
---
**Steps covered:**
1. Install & Import Libraries
2. Load Dataset
3. EDA & Visualizations
4. Text Preprocessing
5. Feature Engineering (BoW, TF-IDF, Word2Vec)
6. ML Models + Hyperparameter Tuning
7. Deep Learning Models (Simple RNN, LSTM, GRU, BiLSTM, Stacked LSTM)
8. Save Best Models (CatBoost & Bidirectional LSTM)
9. Model Comparison Table


## 1. Install Libraries

In [ ]:
!pip install -q missingno catboost lightgbm xgboost gensim nltk wordcloud kagglehub keras-tuner

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, string, os, pickle, warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, SimpleRNN, LSTM, GRU,
                                      Bidirectional, Dense, Dropout)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully ✅")


## 3. Load Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("praveengovi/emotions-dataset-for-nlp")
print("Dataset Path:", path)
print("Files:", os.listdir(path))


In [ ]:
train_df = pd.read_csv(os.path.join(path, "train.txt"), sep=';', names=['Text', 'Emotion'])
test_df  = pd.read_csv(os.path.join(path, "test.txt"),  sep=';', names=['Text', 'Emotion'])
val_df   = pd.read_csv(os.path.join(path, "val.txt"),   sep=';', names=['Text', 'Emotion'])

print("Train:", train_df.shape, "| Test:", test_df.shape, "| Val:", val_df.shape)
train_df.head()


## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Emotion distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (df, title) in zip(axes, [(train_df,'Train'), (test_df,'Test'), (val_df,'Val')]):
    df['Emotion'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette("husl", 6))
    ax.set_title(f'{title} - Emotion Distribution', fontsize=13)
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Text length distribution
train_df['Text_Length'] = train_df['Text'].apply(len)
plt.figure(figsize=(10, 4))
sns.histplot(train_df['Text_Length'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of Text Lengths (Train Set)')
plt.xlabel('Character Length')
plt.ylabel('Frequency')
plt.show()
print("Mean length:", train_df['Text_Length'].mean().round(2))
print("Max length:", train_df['Text_Length'].max())


In [ ]:
# Missing values
import missingno as msno
print("Train missing values:")
print(train_df.isnull().sum())
print("\nTest missing values:")
print(test_df.isnull().sum())


In [ ]:
# WordCloud for each emotion
from wordcloud import WordCloud

emotions = train_df['Emotion'].unique()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, emotion in enumerate(sorted(emotions)):
    text = ' '.join(train_df[train_df['Emotion'] == emotion]['Text'].values)
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap='viridis', max_words=100).generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(f'WordCloud — {emotion.capitalize()}', fontsize=13)
    axes[i].axis('off')

plt.tight_layout()
plt.show()


## 5. Text Preprocessing (Stopwords, Stemming, Lemmatization)

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_text_ml(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return ' '.join(words)

train_df['Clean_Text'] = train_df['Text'].apply(clean_text_ml)
test_df['Clean_Text']  = test_df['Text'].apply(clean_text_ml)
val_df['Clean_Text']   = val_df['Text'].apply(clean_text_ml)

train_df[['Text', 'Clean_Text']].head()


In [ ]:
# Label Encoding
label_encoder = LabelEncoder()
train_df['Emotion_Label'] = label_encoder.fit_transform(train_df['Emotion'])
test_df['Emotion_Label']  = label_encoder.transform(test_df['Emotion'])
val_df['Emotion_Label']   = label_encoder.transform(val_df['Emotion'])

print("Classes:", label_encoder.classes_)

y_train = train_df['Emotion_Label']
y_test  = test_df['Emotion_Label']
y_val   = val_df['Emotion_Label']


## 6. Feature Engineering

In [ ]:
# Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(train_df['Clean_Text'])
X_test_bow  = bow.transform(test_df['Clean_Text'])
print("BoW shape:", X_train_bow.shape)


In [ ]:
# TF-IDF (used for ML models)
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(train_df['Clean_Text'])
X_test_tfidf  = tfidf.transform(test_df['Clean_Text'])
print("TF-IDF shape:", X_train_tfidf.shape)


In [ ]:
# Word2Vec (Gensim)
from gensim.models import Word2Vec

tokenized_train = [text.split() for text in train_df['Clean_Text']]
w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100,
                     window=5, min_count=1, workers=4, epochs=10)
print("Word2Vec vocabulary size:", len(w2v_model.wv))

def get_w2v_vector(text, model, size=100):
    words = text.split()
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(size)

X_train_w2v = np.array([get_w2v_vector(t, w2v_model) for t in train_df['Clean_Text']])
X_test_w2v  = np.array([get_w2v_vector(t, w2v_model) for t in test_df['Clean_Text']])
print("Word2Vec feature shape:", X_train_w2v.shape)


## 7. ML Models (Baseline)

In [ ]:
models = {
    'Logistic Regression':  LogisticRegression(max_iter=1000),
    'Decision Tree':         DecisionTreeClassifier(),
    'Random Forest':         RandomForestClassifier(n_estimators=100),
    'Gradient Boosting':     GradientBoostingClassifier(),
    'AdaBoost':              AdaBoostClassifier(),
    'Bagging':               BaggingClassifier(),
    'KNN':                   KNeighborsClassifier(),
    'XGBoost':               XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', verbosity=0),
    'CatBoost':              CatBoostClassifier(verbose=0),
    'LightGBM':              LGBMClassifier(verbosity=-1),
}

results = []
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    results.append({'Model': name, 'Accuracy': acc})
    print(f"{name}: {acc:.4f}")

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df.style.highlight_max(subset=['Accuracy'], color='lightgreen')


## 8. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Logistic Regression tuning
lr_params = {'C': [0.1, 1, 10], 'max_iter': [500, 1000]}
lr_grid = GridSearchCV(LogisticRegression(), lr_params, cv=3, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train_tfidf, y_train)
lr_best_acc = accuracy_score(y_test, lr_grid.best_estimator_.predict(X_test_tfidf))
print("Logistic Regression best accuracy:", lr_best_acc)


In [ ]:
# XGBoost tuning
xgb_params = {'n_estimators': [100, 200], 'max_depth': [3, 6], 'learning_rate': [0.05, 0.1]}
xgb_grid = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', verbosity=0),
                         xgb_params, cv=3, scoring='accuracy', n_jobs=-1)
xgb_grid.fit(X_train_tfidf, y_train)
xgb_best_acc = accuracy_score(y_test, xgb_grid.best_estimator_.predict(X_test_tfidf))
print("XGBoost best accuracy:", xgb_best_acc)


In [ ]:
# Random Forest tuning
rf_params = {'n_estimators': [100, 200], 'max_depth': [None, 20, 40]}
rf_grid = GridSearchCV(RandomForestClassifier(), rf_params, cv=3, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train_tfidf, y_train)
rf_best_acc = accuracy_score(y_test, rf_grid.best_estimator_.predict(X_test_tfidf))
print("Random Forest best accuracy:", rf_best_acc)


In [ ]:
# LightGBM tuning
lgbm_params = {'n_estimators': [100, 200], 'num_leaves': [31, 64], 'learning_rate': [0.05, 0.1]}
lgbm_grid = GridSearchCV(LGBMClassifier(verbosity=-1), lgbm_params, cv=3, scoring='accuracy', n_jobs=-1)
lgbm_grid.fit(X_train_tfidf, y_train)
lgbm_best_acc = accuracy_score(y_test, lgbm_grid.best_estimator_.predict(X_test_tfidf))
print("LightGBM best accuracy:", lgbm_best_acc)


In [ ]:
# CatBoost — default already best; tune iterations
cat_params = {'iterations': [300, 500, 800], 'learning_rate': [0.05, 0.1], 'depth': [6, 8]}
cat_grid = GridSearchCV(CatBoostClassifier(verbose=0), cat_params, cv=3, scoring='accuracy', n_jobs=-1)
cat_grid.fit(X_train_tfidf, y_train)
cat_best_acc = accuracy_score(y_test, cat_grid.best_estimator_.predict(X_test_tfidf))
print("CatBoost best accuracy:", cat_best_acc)

# Use best CatBoost
best_catboost = cat_grid.best_estimator_


In [ ]:
# Tuned comparison table
tuned_df = pd.DataFrame({
    'Model':          ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost'],
    'Tuned Accuracy': [lr_best_acc, rf_best_acc, xgb_best_acc, lgbm_best_acc, cat_best_acc]
}).sort_values('Tuned Accuracy', ascending=False).reset_index(drop=True)

tuned_df.style.highlight_max(subset=['Tuned Accuracy'], color='lightgreen')


## 9. Save Best ML Model (CatBoost) + TF-IDF + Label Encoder

In [ ]:
os.makedirs('models', exist_ok=True)

with open('models/catboost_model.pkl', 'wb') as f:
    pickle.dump(best_catboost, f)

with open('models/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('models/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("✅ CatBoost model, TF-IDF, and Label Encoder saved to models/")


## 10. Deep Learning — Text Preprocessing

In [ ]:
def clean_text_dl(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    return text

train_df['DL_Text'] = train_df['Text'].apply(clean_text_dl)
test_df['DL_Text']  = test_df['Text'].apply(clean_text_dl)
val_df['DL_Text']   = val_df['Text'].apply(clean_text_dl)

# Tokenizer
MAX_WORDS = 20000
MAX_LEN   = 200

tokenizer_dl = Tokenizer(num_words=MAX_WORDS)
tokenizer_dl.fit_on_texts(train_df['DL_Text'])

X_train_seq  = tokenizer_dl.texts_to_sequences(train_df['DL_Text'])
X_test_seq   = tokenizer_dl.texts_to_sequences(test_df['DL_Text'])
X_val_seq    = tokenizer_dl.texts_to_sequences(val_df['DL_Text'])

X_train_pad  = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post')
X_test_pad   = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post')
X_val_pad    = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding='post')

num_classes  = len(label_encoder.classes_)
y_train_dl   = to_categorical(y_train, num_classes=num_classes)
y_test_dl    = to_categorical(y_test,  num_classes=num_classes)
y_val_dl     = to_categorical(y_val,   num_classes=num_classes)

print("Train pad shape:", X_train_pad.shape)
print("Classes:", num_classes)


## 11. Deep Learning Models

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

def train_and_eval(model, name, epochs=10):
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(X_train_pad, y_train_dl,
              validation_data=(X_val_pad, y_val_dl),
              epochs=epochs, batch_size=64, callbacks=[early_stop], verbose=1)
    loss, acc = model.evaluate(X_test_pad, y_test_dl, verbose=0)
    print(f"\n{name} Test Accuracy: {acc:.4f}")
    return acc


In [ ]:
# Simple RNN
rnn_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    SimpleRNN(64, dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
rnn_acc = train_and_eval(rnn_model, 'Simple RNN')


In [ ]:
# LSTM
lstm_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    LSTM(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
lstm_acc = train_and_eval(lstm_model, 'LSTM')


In [ ]:
# GRU
gru_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    GRU(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
gru_acc = train_and_eval(gru_model, 'GRU')


In [ ]:
# Bidirectional LSTM  ← BEST DL MODEL
bilstm_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.3)),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
bilstm_acc = train_and_eval(bilstm_model, 'Bidirectional LSTM', epochs=15)


In [ ]:
# Stacked LSTM
stacked_lstm = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    LSTM(128, return_sequences=True, dropout=0.3),
    LSTM(64, dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
stacked_acc = train_and_eval(stacked_lstm, 'Stacked LSTM')


## 12. Save Best DL Model (Bidirectional LSTM) + Tokenizer

In [ ]:
bilstm_model.save('models/bilstm_model.h5')

with open('models/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer_dl, f)

print("✅ Bidirectional LSTM model and Tokenizer saved to models/")


## 13. Final Comparison Table — All Models

In [ ]:
dl_results_df = pd.DataFrame([
    {'Model': 'Simple RNN',          'Type': 'Deep Learning', 'Accuracy': rnn_acc},
    {'Model': 'LSTM',                'Type': 'Deep Learning', 'Accuracy': lstm_acc},
    {'Model': 'GRU',                 'Type': 'Deep Learning', 'Accuracy': gru_acc},
    {'Model': 'Bidirectional LSTM',  'Type': 'Deep Learning', 'Accuracy': bilstm_acc},
    {'Model': 'Stacked LSTM',        'Type': 'Deep Learning', 'Accuracy': stacked_acc},
    {'Model': 'Logistic Regression', 'Type': 'ML (TF-IDF)',   'Accuracy': lr_best_acc},
    {'Model': 'Random Forest',       'Type': 'ML (TF-IDF)',   'Accuracy': rf_best_acc},
    {'Model': 'XGBoost',             'Type': 'ML (TF-IDF)',   'Accuracy': xgb_best_acc},
    {'Model': 'LightGBM',            'Type': 'ML (TF-IDF)',   'Accuracy': lgbm_best_acc},
    {'Model': 'CatBoost',            'Type': 'ML (TF-IDF)',   'Accuracy': cat_best_acc},
]).sort_values('Accuracy', ascending=False).reset_index(drop=True)

dl_results_df.style.highlight_max(subset=['Accuracy'], color='lightgreen')


In [ ]:
# Bar chart comparison
plt.figure(figsize=(14, 6))
colors = ['#6a11cb' if t == 'Deep Learning' else '#2575fc' for t in dl_results_df['Type']]
bars = plt.barh(dl_results_df['Model'], dl_results_df['Accuracy'], color=colors)
plt.xlabel('Accuracy', fontsize=13)
plt.title('Model Comparison — Emotion Detection', fontsize=15)
plt.xlim(0.3, 1.0)

for bar, acc in zip(bars, dl_results_df['Accuracy']):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{acc:.4f}', va='center', fontsize=10)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#6a11cb', label='Deep Learning'),
                   Patch(facecolor='#2575fc', label='ML (TF-IDF)')]
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=150)
plt.show()
print("\n🏆 Best Model:", dl_results_df.iloc[0]['Model'],
      f"| Accuracy: {dl_results_df.iloc[0]['Accuracy']:.4f}")


## ✅ Summary

| Model | Accuracy |
|---|---|
| **Bidirectional LSTM** | **~89.7%** |
| CatBoost (tuned) | ~86.75% |
| Logistic Regression (tuned) | ~84.70% |

**Best model: Bidirectional LSTM** saved to `models/bilstm_model.h5`

Run the Streamlit app with:
```bash
streamlit run app.py
```
